In [ ]:
!pip install sentence-transformers
!pip install chromadb
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

# Project Description

This project implements a simple NLP system that allows users to ask questions.
The system converts text into embeddings using a Sentence Transformer model.
These embeddings are stored in a vector database (ChromaDB).
When a user asks a question, the system converts it into a vector and searches
for the most semantically similar answers.

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
data = pd.read_csv("qa_chunks.csv")
texts = data["text"].tolist()
print("Number of texts:", len(texts))

Number of texts: 1881


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Convert texts to embeddings
embeddings = model.encode(texts)
print("Embeddings created successfully")

Embeddings created successfully


In [ ]:
# Create Vector Database
client = chromadb.Client()
collection = client.create_collection(name="qa_collection")

In [ ]:
# Store vectors in database
for i, text in enumerate(texts):
    collection.add(
        documents=[text],
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )
print("Vector Database created successfully!")

Vector Database created successfully!


In [ ]:
# Take question from user
query = input("Enter your question: ")
# Convert question to vector
query_vector = model.encode([query])

Enter your question: Where is my order?


In [ ]:
# Search in Vector Database
results = collection.query(
    query_embeddings=query_vector,
    n_results=3) #n_results = 1 بقولو هاتلي افضل 3

In [ ]:
print("\nBest Answers:\n")
for answer in results["documents"][0]:
    print(answer)


Best Answers:

Question: I ordered a toy from your website, and I haven't received it yet. Can you tell me where my order is?
Answer: The order is currently in transit and is expected to arrive within the next two days.
Question: Can I know the current location of my order?
Answer: Your order is currently in transit and is expected to arrive at our local facility within the next 24 hours.
Question: Where is the invoice for the order?
Answer: The invoice was sent to the customer's spam folder on the 7th of the month. They have now found it.
